# FinReasoning AI Gradio Demo (Merged Model)

This notebook loads the merged model from `outputs/merged_model` and launches a public Gradio app (`share=True`) for financial question answering.


## Setup

Mount Drive, infer workspace, clone/pull repo, and install minimal dependencies.

In [ ]:
from pathlib import Path
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_FALLBACK = '/content/drive/MyDrive/FinReasoningAI'

def _infer_notebook_workspace():
    """Parent folder of FinReasoningAI_Colab.ipynb on Drive (depth-limited, quick)."""
    root = Path('/content/drive/MyDrive')
    if not root.is_dir():
        return None
    candidates = []
    if (root / 'FinReasoningAI_Colab.ipynb').is_file():
        candidates.append(root.resolve())
    for child in sorted(root.iterdir()):
        if not child.is_dir():
            continue
        if (child / 'FinReasoningAI_Colab.ipynb').is_file():
            candidates.append(child.resolve())
        nested = child / 'FinReasoningAI'
        if nested.is_dir() and (nested / 'FinReasoningAI_Colab.ipynb').is_file():
            candidates.append(nested.resolve())
    uniq = []
    seen = set()
    for c in candidates:
        s = str(c)
        if s not in seen:
            seen.add(s)
            uniq.append(c)
    if len(uniq) == 1:
        return str(uniq[0])
    if len(uniq) > 1:
        print('[WARN] Multiple FinReasoningAI_Colab.ipynb paths on Drive; using DRIVE_FALLBACK.')
    return None

DRIVE_BASE = _infer_notebook_workspace() or DRIVE_FALLBACK
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive workspace: {DRIVE_BASE}')


In [ ]:
REPO_URL = 'https://github.com/juankim834/FinReasoningAI.git'

import os
import sys

WORKSPACE = DRIVE_BASE
if os.path.isdir(os.path.join(WORKSPACE, '.git')):
    PROJECT_DIR = WORKSPACE
else:
    PROJECT_DIR = os.path.join(WORKSPACE, 'FinReasoningAI')

if not os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
    os.makedirs(WORKSPACE, exist_ok=True)
    print(f'Cloning {REPO_URL} -> {PROJECT_DIR}')
    get_ipython().system(f'git clone {REPO_URL} {PROJECT_DIR}')
else:
    print(f'Repo already at {PROJECT_DIR}. Pulling latest...')
    get_ipython().system(f'cd {PROJECT_DIR} && git pull')

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')


In [ ]:
"""
Install only required dependencies (no torch reinstall).
Hard requirement: bitsandbytes >= 0.44.0.
"""

import importlib.metadata
import subprocess
import sys

REQUIRED = {
    'transformers': '4.41.0',
    'peft': '0.10.0',
    'bitsandbytes': '0.44.0',
    'accelerate': '0.30.0',
    'gradio': '4.0.0',
    'vllm': '0.4.0',
}

def _parse(v):
    out = []
    for p in v.split('.'):
        if p.isdigit():
            out.append(int(p))
        else:
            break
    while len(out) < 3:
        out.append(0)
    return tuple(out[:3])

def _installed(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None

to_install = []
for pkg, min_ver in REQUIRED.items():
    cur = _installed(pkg)
    if cur is None or _parse(cur) < _parse(min_ver):
        to_install.append(f'{pkg}>={min_ver}')
        status = 'MISSING' if cur is None else f'upgrade {cur} -> >= {min_ver}'
    else:
        status = f'ok ({cur})'
    print(f'{pkg:<14} {status}')

if to_install:
    print(f'\nInstalling {len(to_install)} package(s): {to_install}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + to_install)
    print('Install complete.')
else:
    print('\nAll required packages already satisfy minimum versions.')

bnb_ver = importlib.metadata.version('bitsandbytes')
if _parse(bnb_ver) < _parse('0.44.0'):
    raise RuntimeError(
        f'bitsandbytes {bnb_ver} is installed but >= 0.44.0 is required.\n'
        "Fix: pip install -U 'bitsandbytes>=0.44.0' then Runtime > Restart session."
    )

import torch
print(f'\nPyTorch version (unchanged): {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / (1024**3):.1f} GB')


In [ ]:
# Ensure outputs are symlinked to Drive.
from pathlib import Path
import shutil
import os

def ensure_drive_symlink(rel_path: str):
    drive_path = Path(DRIVE_BASE) / rel_path
    local_path = Path(PROJECT_DIR) / rel_path
    drive_path.mkdir(parents=True, exist_ok=True)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.is_symlink():
        print(f'[OK] Symlink exists: {local_path} -> {local_path.resolve()}')
        return

    if local_path.exists():
        if local_path.is_dir():
            shutil.copytree(local_path, drive_path, dirs_exist_ok=True)
            shutil.rmtree(local_path)
        else:
            shutil.copy2(local_path, drive_path)
            local_path.unlink()

    os.symlink(drive_path, local_path)
    print(f'[OK] Linked: {local_path} -> {drive_path}')

for _rel in ['outputs/merged_model', 'outputs/sft_qlora']:
    ensure_drive_symlink(_rel)


In [ ]:
# Optional Hugging Face login with fallback.
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in using Colab secret HF_TOKEN.')
except Exception:
    print('HF_TOKEN secret not found. Falling back to interactive login...')
    login()


## Model Load

Load the merged model from `outputs/merged_model` in BF16 with automatic device mapping.

In [ ]:
from pathlib import Path
import torch
from src.inference.generate import generate_answer, build_prompt, _vllm_generate_texts
from src.model.load_model import DEFAULT_MODEL_ID, load_vllm_model_and_tokenizer

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU runtime is required for this demo.')

MERGED_MODEL_DIR = Path('outputs/merged_model')
MODEL_PATH = str(MERGED_MODEL_DIR) if MERGED_MODEL_DIR.exists() and any(MERGED_MODEL_DIR.iterdir()) else DEFAULT_MODEL_ID

print(f'Loading vLLM inference engine from: {MODEL_PATH}')
model, tokenizer = load_vllm_model_and_tokenizer(
    model_id=MODEL_PATH,
    max_model_len=2048,
    gpu_memory_utilization=0.90,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('[OK] vLLM inference engine loaded.')


## Keep-Alive

Start a daemon thread to keep the Colab runtime active during the demo session.

In [ ]:
import threading
import time

def _keep_alive_loop():
    while True:
        # Lightweight heartbeat every 60 seconds.
        time.sleep(60)

keep_alive_thread = threading.Thread(target=_keep_alive_loop, daemon=True)
keep_alive_thread.start()
print('[OK] Keep-alive daemon thread started.')


## Demo Launch

Build and launch the Gradio interface with Direct Answer, Chain-of-Thought, and Self-Consistency modes.

In [ ]:
import re
import time
import pandas as pd
import torch
import gradio as gr

THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

APPLE_CONTEXT_PLACEHOLDER = (
    "Apple 2022 10-K excerpt:\n"
    "Net sales were $394.3 billion in 2022 and $365.8 billion in 2021.\n"
    "Research and development expense was $26.3 billion in 2022.\n"
    "Operating cash flow was $122.2 billion and capital expenditures were $10.7 billion."
)

EXAMPLES = [
    [
        'What was Apple\'s year-over-year revenue growth from 2021 to 2022?',
        'Net sales were $394.3 billion in 2022 and $365.8 billion in 2021.',
        'Direct Answer',
    ],
    [
        'What percentage of revenue did Apple spend on R&D in 2022?',
        'Revenue was $394.3 billion and R&D expense was $26.3 billion in 2022.',
        'Chain-of-Thought',
    ],
    [
        'Estimate Apple\'s 2022 free cash flow using provided values.',
        'Operating cash flow was $122.2 billion and capex was $10.7 billion.',
        'Self-Consistency (N=8)',
    ],
]

def _capture_raw_cot(question: str, context: str, max_new_tokens: int = 400) -> str:
    """Generate raw CoT trace including <think> tags for demo transparency."""
    prompt = build_prompt(question=question, context=context, use_cot=True)
    outputs = _vllm_generate_texts(
        model=model,
        prompts=[prompt],
        temperature=0.0,
        max_new_tokens=max_new_tokens,
        n=1,
    )
    return outputs[0][0].strip() if outputs and outputs[0] else ''

def _estimate_ttft_ms(question: str, context: str, use_cot: bool) -> float:
    prompt = build_prompt(question=question, context=context, use_cot=use_cot)
    t0 = time.perf_counter()
    _ = _vllm_generate_texts(
        model=model,
        prompts=[prompt],
        temperature=0.0,
        max_new_tokens=1,
        n=1,
    )
    t1 = time.perf_counter()
    return (t1 - t0) * 1000.0

def _benchmark_table(ttft_ms, total_ms, tokens_generated, tps, peak_vram_mb):
    rows = [
        ['Time to first token (ms)', f'{ttft_ms:.2f}'],
        ['Total generation time (ms)', f'{total_ms:.2f}'],
        ['Tokens generated', str(tokens_generated)],
        ['Tokens per second', f'{tps:.2f}'],
        ['Peak VRAM used (MB)', f'{peak_vram_mb:.2f}'],
    ]
    return pd.DataFrame(rows, columns=['Metric', 'Value'])

def run_finreasoning_demo(question: str, context: str, mode: str):
    question = (question or '').strip()
    context = (context or '').strip()

    if not question:
        empty_df = _benchmark_table(0.0, 0.0, 0, 0.0, 0.0)
        return 'Please provide a question.', 'CoT not enabled for this mode.', empty_df

    use_cot = mode == 'Chain-of-Thought'
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    ttft_ms = _estimate_ttft_ms(question=question, context=context, use_cot=use_cot)

    start = time.perf_counter()

    if mode == 'Direct Answer':
        answer = generate_answer(
            model=model,
            tokenizer=tokenizer,
            question=question,
            context=context,
            use_cot=False,
            grounding_check=True,
            temperature=0.0,
        )
        cot_text = 'CoT not enabled for this mode.'
        text_for_token_count = answer
    elif mode == 'Chain-of-Thought':
        # Requirement: use generate_answer for official answer generation in CoT mode.
        answer = generate_answer(
            model=model,
            tokenizer=tokenizer,
            question=question,
            context=context,
            use_cot=True,
            max_new_tokens=400,
            grounding_check=True,
            temperature=0.0,
        )
        # Also capture raw trace (with <think> tags) for the CoT tab.
        cot_text = _capture_raw_cot(question=question, context=context, max_new_tokens=400)
        if '</think>' not in cot_text and not THINK_RE.search(cot_text):
            cot_text = cot_text + '\n\n[WARN] No explicit <think> tags were emitted in this run.'
        text_for_token_count = cot_text
    elif mode == 'Self-Consistency (N=8)':
        answer = generate_answer(
            model=model,
            tokenizer=tokenizer,
            question=question,
            context=context,
            use_cot=False,
            self_consistency_n=8,
            temperature=0.7,
            min_confidence=0.30,
            grounding_check=True,
        )
        cot_text = 'CoT not enabled for this mode.'
        text_for_token_count = answer
    else:
        raise ValueError(f'Unsupported mode: {mode}')

    end = time.perf_counter()
    total_ms = (end - start) * 1000.0

    tokens_generated = len(tokenizer.encode(text_for_token_count, add_special_tokens=False))
    tps = (tokens_generated / (total_ms / 1000.0)) if total_ms > 0 else 0.0
    peak_vram_mb = (
        torch.cuda.max_memory_allocated() / (1024**2)
        if torch.cuda.is_available()
        else 0.0
    )

    bench_df = _benchmark_table(ttft_ms, total_ms, tokens_generated, tps, peak_vram_mb)
    return answer, cot_text, bench_df

with gr.Blocks(title='FinReasoning AI - Financial QA Demo') as demo:
    gr.Markdown('# FinReasoning AI - Financial QA Demo')
    gr.Markdown('Ask grounded financial QA questions using the fine-tuned merged Qwen2.5-14B model.')

    with gr.Row():
        question_input = gr.Textbox(label='Question', placeholder='Enter your financial question...')

    context_input = gr.Textbox(
        label='Financial Context',
        lines=8,
        placeholder=APPLE_CONTEXT_PLACEHOLDER,
    )

    mode_input = gr.Radio(
        choices=['Direct Answer', 'Chain-of-Thought', 'Self-Consistency (N=8)'],
        value='Direct Answer',
        label='Inference Mode',
    )

    run_btn = gr.Button('Run Inference', variant='primary')

    with gr.Tabs():
        with gr.Tab('Answer'):
            answer_output = gr.Textbox(label='Final Answer', lines=6)
        with gr.Tab('Chain-of-Thought'):
            cot_output = gr.Textbox(label='Reasoning Trace', lines=14)
        with gr.Tab('Efficiency Benchmark'):
            bench_output = gr.Dataframe(
                headers=['Metric', 'Value'],
                datatype=['str', 'str'],
                row_count=5,
                col_count=(2, 'fixed'),
                wrap=True,
                label='Benchmark',
            )

    run_btn.click(
        fn=run_finreasoning_demo,
        inputs=[question_input, context_input, mode_input],
        outputs=[answer_output, cot_output, bench_output],
    )

    gr.Examples(
        examples=EXAMPLES,
        inputs=[question_input, context_input, mode_input],
        outputs=[answer_output, cot_output, bench_output],
        fn=run_finreasoning_demo,
        cache_examples=False,
    )

demo.launch(share=True, debug=False)
